In [18]:
import json
import re
import warnings
import pickle
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from sentence_transformers import SentenceTransformer

In [19]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [20]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

# DAPT 훈련 결과 경로
lora_model_path = "./dapt_model/checkpoint-552/"

# 1) DAPT 훈련 모델 로드 (trainable=True 중요!)
model = AutoPeftModelForCausalLM.from_pretrained(
    lora_model_path,
    device_map="auto",
    torch_dtype="auto",
    is_trainable=True,
)
model.train()
model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# 2) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(lora_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 확인
from peft import PeftModel
if isinstance(model, PeftModel):
    model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


In [ ]:
import re
import pandas as pd
from pathlib import Path

import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig  # ⬅ DataCollatorForCompletionOnlyLM 대신 커스텀 사용

# 0) 학습 데이터 준비 (엑셀 -> chat 텍스트)
EXCEL_PATH = "HCU_filter/HCU_filter_merged.xlsx"
OUT_DIR = "./lora_outputs_hcu_ft"  # 추가 파인튜닝 결과 저장 경로

input_columns = [
    "ID", "검증 목적", "시험 전 조건", "시험 방법", "판정 조건",
    "시험 전 조건 (변수명 개수)", "시험 방법 (변수명 개수)", "판정 조건 (변수명 개수)"
]
target_columns = ["시험 전 조건 (변수명)", "시험 방법 (변수명)", "판정 조건 (변수명)"]

BRACKET_TAIL = re.compile(r"(?:\[[^\]]*\])+$")
# 하이픈/콤마/세미콜론/불릿/슬래시 등을 변수 구분자로 간주
SEP = re.compile(r"(?:\s*-\s*|[\s,;/·•]+)")

def _clean_var(x):
    if x is None:
        return ""
    x = str(x).strip().replace("<|eot_id|>", "")
    x = BRACKET_TAIL.sub("", x).strip()
    return x

def _normalize_target_block(text):
    """학습 타겟을 한 줄당 하나의 변수로 표준화 + 하이픈/콤마 연결 케이스 분리."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return []
    items = []
    for line in str(text).splitlines():
        for tok in SEP.split(line.strip()):   # ⬅ 연결기호로 붙어 있으면 분리
            t = _clean_var(tok)
            if t and t != "-":
                items.append(t)
    # 중복 제거(순서 보존)
    seen, out = set(), []
    for v in items:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

def build_user_prompt(row):
    blocks = []
    for key in input_columns:
        v = "" if pd.isna(row.get(key)) else str(row.get(key))
        blocks.append(f"{key}:\n{v}")
    return "\n\n".join(blocks)

def build_assistant_answer(row):
    pre = _normalize_target_block(row.get("시험 전 조건 (변수명)"))
    scr = _normalize_target_block(row.get("시험 방법 (변수명)"))
    out = _normalize_target_block(row.get("판정 조건 (변수명)"))

    def blk(title, arr):
        return f"{title}:\n" + ("\n".join(arr) if arr else "")

    return "\n\n".join([
        blk("시험 전 조건 (변수명)", pre),
        blk("시험 방법 (변수명)", scr),
        blk("판정 조건 (변수명)", out),
    ])

# 🔒 프롬프트 제약 강화
SYSTEM_PROMPT = (
    "너는 자율주행 차량 제어 도메인의 변수명 생성 전문가다. "
    "아래 입력(검증 목적/시험 전 조건/시험 방법/판정 조건과 각 변수명 개수)을 바탕으로 "
    "섹션별 변수명을 정확히 생성하라.\n\n"
    "출력 형식은 반드시 아래와 같아야 한다(헤더/개행 포함, 다른 텍스트 금지):\n\n"
    "시험 전 조건 (변수명):\n<변수명만 줄바꿈으로 나열>\n\n"
    "시험 방법 (변수명):\n<변수명만 줄바꿈으로 나열>\n\n"
    "판정 조건 (변수명):\n<변수명만 줄바꿈으로 나열>\n\n"
    "변수명 생성 제약 조건은 다음과 같다.\n"
    "1) 각 변수명은 한 줄에 정확히 하나만 작성한다.\n"
    "2) '-', ',', '/', '·', '•', 'and', 'or' 등 어떤 구분자도 포함하지 않는다.\n"
    "3) 변수명 외의 설명/문장/번호매기기(예: '1.', '-')는 절대 포함하지 않는다.\n"
)

df_train = pd.read_excel(EXCEL_PATH)

# 타깃이 전부 비어있거나 '-'이면 제외
def _has_any_target(row):
    for c in target_columns:
        v = row.get(c)
        if isinstance(v, str) and v.strip() not in ("", "-"):
            return True
    return False

df_train = df_train[df_train.apply(_has_any_target, axis=1)].reset_index(drop=True)

# chat 예시 리스트 생성
sft_list = []
for _, r in df_train.iterrows():
    sft_list.append({
        "system": SYSTEM_PROMPT,
        "user": build_user_prompt(r),
        "assistant": build_assistant_answer(r),
    })

# text 컬럼 생성
def to_text(ex):
    return {
        "text": (
            f"<<SYS>>\n{ex['system']}\n<</SYS>>\n\n"
            f"[USER]\n{ex['user']}\n\n"
            f"[ASSISTANT]\n{ex['assistant']}"
        )
    }

train_ds = Dataset.from_list(sft_list)
train_ds = train_ds.map(to_text)
remove_cols = [c for c in train_ds.column_names if c != "text"]
if remove_cols:
    train_ds = train_ds.remove_columns(remove_cols)

# rain / test 분리
split = train_ds.train_test_split(test_size=80, seed=52)
train_ds = split["train"]
test_ds = split["test"]
print(f"train={len(train_ds)}, test={len(test_ds)}")

# 0) no_grad/eval/캐시 설정 정리
torch.set_grad_enabled(True)
model.train()
model.config.use_cache = False

# 1) 입력 임베딩에도 grad 연결 (일부 환경에서 필요)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# 2) 토크나이저 pad 토큰 확인
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3) LoRA 파라미터 학습 가능 여부 출력(선택)
try:
    from peft import PeftModel
    if isinstance(model, PeftModel):
        model.print_trainable_parameters()
except Exception as e:
    print("PEFT trainable 확인 중 경고:", e)

# 커스텀 collator: padding/truncation 보장 + [ASSISTANT]\n 이후만 라벨
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class CompletionOnlyCollator:
    tokenizer: Any
    response_template: str = "[ASSISTANT]\n"

    def __post_init__(self):
        self.template_ids = self.tokenizer.encode(
            self.response_template, add_special_tokens=False
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __call__(self, examples: List[Dict[str, str]]) -> Dict[str, torch.Tensor]:
        texts = [ex["text"] for ex in examples]
        batch = self.tokenizer(
            texts,
            padding=True,          
            truncation=True,        
            return_tensors="pt"
        )
        input_ids = batch["input_ids"]
        labels = input_ids.clone()
        labels[batch["attention_mask"] == 0] = -100  # pad 라벨 마스킹

        # [ASSISTANT]\n 템플릿 이후만 학습
        temp = torch.tensor(self.template_ids, dtype=torch.long)
        tlen = temp.size(0)
        for i in range(input_ids.size(0)):
            seq = input_ids[i]
            start_idx = -1
            for j in range(0, seq.size(0) - tlen + 1):
                if torch.equal(seq[j:j+tlen], temp):
                    start_idx = j + tlen
                    break
            if start_idx == -1:
                labels[i, :] = -100
            else:
                labels[i, :start_idx] = -100

        batch["labels"] = labels
        return batch

collator = CompletionOnlyCollator(
    tokenizer=tokenizer,
    response_template="[ASSISTANT]\n",
)

# SFT 설정 (리소스/선호에 맞게 조절)
MAX_SEQ_LEN = 4096
sft_args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,                   # ⬅ completion-only collator와는 False
    gradient_checkpointing=False,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    bf16=torch.cuda.is_available(),
    fp16=False,
    # 평가/로깅
    eval_strategy="steps",
    eval_steps=20,
    logging_steps=20,
    # text 컬럼 보존
    remove_unused_columns=False,
    dataset_text_field="text",
    save_steps=250,
    save_total_limit=2,
    save_safetensors=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,  # test를 eval로 사용
    dataset_text_field="text",
    data_collator=collator,  # 커스텀 collator
    args=sft_args,
)

trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"[FT] Continued LoRA saved to: {OUT_DIR}")

In [21]:
from transformers import AutoTokenizer
from peft import PeftModel, AutoPeftModelForCausalLM
import torch

# LoRA 훈련 결과 경로 (예: ./lora_outputs/fold_0)
lora_model_path = './lora_outputs_hcu_ft/checkpoint-500' 

# LoRA 기반 모델 로드
model = AutoPeftModelForCausalLM.from_pretrained(
    lora_model_path,
    device_map="auto",
    torch_dtype="auto"
)

# 토크나이저 로드 (base 모델에서 가져옴)
base_model_id = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [22]:
with open('input-HCU-RAG.json', 'r') as f:
    data = json.load(f)

In [23]:
def build_indexes(data, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    
    model_rag = SentenceTransformer(model_name)
    db = []

    for item in data:
        item_id = item['ID'].strip()
        item_text = item['text'].strip()

        emb = model_rag.encode([item_text], convert_to_numpy=True, show_progress_bar=False)
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)

        db.append({"ID": item_id, "text": item_text, "emb": emb[0]})
        
    return model_rag, db

In [24]:
def retrieve_similar_texts(query, input_id, model_rag, db, k=5):
    filtered_items = []
    embeddings_list = []
    
    for item in db:
        if item['ID'] != input_id:
            filtered_items.append(item)
            embeddings_list.append(item['emb'])

    if not filtered_items:
        return []

    embeddings_matrix = np.vstack(embeddings_list)

    q_emb = model_rag.encode([query], convert_to_numpy=True, show_progress_bar=False)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)

    sims = (q_emb @ embeddings_matrix.T)[0]

    top_k_indices = sims.argsort()[::-1][:k]

    results = []
    for i in top_k_indices:
        item = filtered_items[i]
        score = float(sims[i])
        results.append({"ID": item["ID"], "text": item["text"], "similarity": score})

    return results

In [25]:
model_rag, db = build_indexes(data)

In [26]:
def build_example_prompt(example_rows, input_columns, target_columns):
    prompt = ""

    idx = 1

    for _, row in example_rows.iterrows():
        prompt += f"[입력 예시 {idx}]\n"
        for col in input_columns:
            prompt += f"{col}:\n{row[col]}\n\n"
        
        # for col in condition_columns:
        #     prompt += f"{col}:\n{row[col]}\n\n"
        
        prompt += f"[예측]\n"

        for col in target_columns:
            value = str(row[col]).strip()
            prompt += f"{col}:\n{value}\n\n"

        idx += 1

    return prompt

In [27]:
def build_user_prompt(row, input_columns):
    """
    사용자의 실제 입력 프롬프트 생성
    """
    return "\n\n".join(f"{col}:\n{row[col]}" for col in input_columns)

In [28]:
df = pd.read_excel('HCU_filter/HCU_filter_merged.xlsx')

In [ ]:
import re
from tqdm import tqdm

SECTION_KEYS = [
    "시험 전 조건 (변수명)",
    "시험 방법 (변수명)",
    "판정 조건 (변수명)",
]

input_columns = [
    "ID", "검증 목적", "시험 전 조건", "시험 방법", "판정 조건",
    "시험 전 조건 (변수명 개수)", "시험 방법 (변수명 개수)", "판정 조건 (변수명 개수)"
]
target_columns = ["시험 전 조건 (변수명)", "시험 방법 (변수명)", "판정 조건 (변수명)"]

# 대괄호 안의 어떤 내용이든 제거:  [ ... ] -> ""  (여러 개도 모두 제거)
BRACKET_RE = re.compile(r"\[[^\[\]]*\]")

def normalize_var(name: str) -> str:
    """변수명에서 대괄호 구간([...]) 제거하고 좌우 공백 정리."""
    return BRACKET_RE.sub("", name).strip()

def clean_split_lines(text: str, normalize: bool = True):
    """줄바꿈으로 분리하고 공백/빈줄/'-' 제거 (+ 선택적으로 정규화)."""
    if text is None:
        return []
    parts = re.split(r"[\r\n]+", str(text))
    items = [p.strip() for p in parts if p and p.strip() and p.strip() != "-"]
    if normalize:
        items = [normalize_var(x) for x in items]
        # 정규화로 빈 문자열이 될 수도 있으니 필터링
        items = [x for x in items if x]
    # 중복 제거(순서 보존)
    seen = set()
    uniq = []
    for x in items:
        if x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq

def extract_predicted_sections(text: str):
    """
    생성 결과 text에서 섹션별 변수 리스트 파싱.
    섹션 헤더는 정확히 아래 3개를 기대:
      - '시험 전 조건 (변수명):'
      - '시험 방법 (변수명):'
      - '판정 조건 (변수명):'
    """
    preds = {k: [] for k in SECTION_KEYS}
    pattern = (
        r"(시험 전 조건 \(변수명\):\s*(?P<pre>.*?))(?=(\n\n시험 방법 \(변수명\):|\Z))"
        r"|"
        r"(시험 방법 \(변수명\):\s*(?P<script>.*?))(?=(\n\n판정 조건 \(변수명\):|\Z))"
        r"|"
        r"(판정 조건 \(변수명\):\s*(?P<out>.*))"
    )
    for m in re.finditer(pattern, text, flags=re.DOTALL):
        if m.group("pre") is not None:
            preds["시험 전 조건 (변수명)"] = clean_split_lines(m.group("pre"), normalize=True)
        if m.group("script") is not None:
            preds["시험 방법 (변수명)"] = clean_split_lines(m.group("script"), normalize=True)
        if m.group("out") is not None:
            preds["판정 조건 (변수명)"] = clean_split_lines(m.group("out"), normalize=True)
    return preds

def prf_from_sets(gt_list, pred_list):
    """집합 기반 precision/recall/f1 계산."""
    gt, pr = set(gt_list), set(pred_list)
    tp = len(gt & pr)
    fp = len(pr - gt)
    fn = len(gt - pr)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = (2*prec*rec)/(prec+rec) if (prec+rec) > 0 else 0.0
    return prec, rec, f1, tp, fp, fn

df = df.sample(n=50, random_state=1234).reset_index(drop=True)
selected_columns = input_columns + target_columns
df = df[selected_columns]

# SECTION_KEYS 모두 비었거나 '-' 또는 NaN인 행은 제외
def is_valid_row(row):
    for key in SECTION_KEYS:
        val = row.get(key, None)
        if pd.isna(val):
            continue
        if str(val).strip() == "" or str(val).strip() == "-":
            continue
        # 하나라도 유효한 값이 있으면 True
        return True
    # 전부 NaN/빈칸/'-'인 경우 False
    return False

df = df[df.apply(is_valid_row, axis=1)].reset_index(drop=True)

SYSTEM_PROMPT = (
    "너는 자율주행 차량 제어 도메인의 변수명 생성 전문가다. "
    "아래 입력(검증 목적/시험 전 조건/시험 방법/판정 조건과 각 변수명 개수)을 바탕으로 "
    "섹션별 변수명을 정확히 생성하라.\n\n"
    "출력 형식을 반드시 정확히 지켜라(헤더/개행 포함):\n\n"
    "시험 전 조건 (변수명):\n<변수명1>\n<변수명2>\n...\n\n"
    "시험 방법 (변수명):\n<변수명1>\n<변수명2>\n...\n\n"
    "판정 조건 (변수명):\n<변수명1>\n<변수명2>\n...\n\n"
    "규칙:\n"
    "- 각 변수명은 한 줄에 정확히 하나만 작성한다.\n"
    "- '-', ',', '/', '·', '•', 'and', 'or' 등 어떤 구분자도 사용하지 않는다.\n"
)

# 루프: 생성 + 평가
results = []

# 집계용 
per_row_scores = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    test_prompt = build_user_prompt(row, input_columns)
    ID = test_prompt.split("ID:\n")[-1].split("\n\n")[0]

    # RAG 검색
    top_results = retrieve_similar_texts(test_prompt, ID, model_rag, db, k=5)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # few-shot 예시 추가 (q/a 분리 가드)
    for p in top_results:
        parts = p['text'].split("\n\n시험 전 조건 (변수명):\n", 1)
        if len(parts) != 2:
            continue
        q = parts[0]
        a = "시험 전 조건 (변수명):\n" + parts[1]
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": a})

    messages.append({"role": "user", "content": test_prompt})

    # 입력/생성 분리 가능한 텍스트 구성
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 입력 토큰 길이 기억
    input_len = model_inputs.input_ids.shape[1]

    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True
        )

    # 생성부분만 추출
    gen_ids = generated.sequences[0][input_len:]
    response_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    results.append(response_text)

    # GT / Pred 파싱 & F1 계산 
    gt_map = {
        "시험 전 조건 (변수명)": clean_split_lines(row.get("시험 전 조건 (변수명)", None)),
        "시험 방법 (변수명)":   clean_split_lines(row.get("시험 방법 (변수명)", None)),
        "판정 조건 (변수명)":   clean_split_lines(row.get("판정 조건 (변수명)", None)),
    }

    pred_map = extract_predicted_sections(response_text)

    row_scores = {}
    for key in SECTION_KEYS:
        prec, rec, f1, _, _, _ = prf_from_sets(gt_map[key], pred_map.get(key, []))
        row_scores[f"P_{key}"] = prec
        row_scores[f"R_{key}"] = rec
        row_scores[f"F1_{key}"] = f1

    per_row_scores.append(row_scores)

    # 추가: GT vs Pred print
    print(f"\n=== Row {row['ID']} ===")
    for key in SECTION_KEYS:
        print(f"[{key}]")
        print("GT   :", gt_map[key])
        print("Pred :", pred_map.get(key, []))
    print("-" * 40)

# 집계 출력 (F1만)
macro_F1 = {}
for key in SECTION_KEYS:
    vals = [r[f"F1_{key}"] for r in per_row_scores]
    macro_F1[key] = sum(vals)/len(vals) if vals else 0.0

print("\n=== F1 Summary ===")
for key in SECTION_KEYS:
    print(f"{key} (F1 score): {macro_F1[key]:.4f}")